In [8]:
import os
import json
import time
from faster_whisper import WhisperModel

# 1. Configuración de directorios
audio_dir = os.path.join("..", "data", "transcripts")
transcript_dir = os.path.join("..", "data", "transcripts")
os.makedirs(transcript_dir, exist_ok=True)

# 2. Carga del modelo)
# device="cuda" y compute_type="float16"
model = WhisperModel("large-v3", device="cuda", compute_type="float16")
print("Modelo cargado correctamente. Iniciando procesamiento...\n")

# 3. Bucle de transcripción
for file in os.listdir(audio_dir):
    if file.endswith(".mp3"):
        audio_path = os.path.join(audio_dir, file)
        json_file = f"{os.path.splitext(file)[0]}.json"
        json_path = os.path.join(transcript_dir, json_file)
        
        # Sistema de caché: evitamos reprocesar si el JSON ya existe
        if os.path.exists(json_path):
            print(f"Saltando '{file}', ya existe su transcripción.")
            continue
            
        print(f"Transcribiendo: {file}...")
        start_time = time.time()
        
        # beam_size=5 mejora la coherencia del texto a costa de un poco de velocidad
        # condition_on_previous_text=False evita que el modelo entre en bucles de alucinación con el ruido del estadio
        segments, info = model.transcribe(
            audio_path, 
            language="es",
            beam_size=5,
            condition_on_previous_text=False 
        )
        
        print(f"Detectado idioma '{info.language}' con probabilidad {info.language_probability:.2f}")
        
        # faster-whisper devuelve un generador, iteramos para extraer los datos
        transcription_data = []
        for segment in segments:
            transcription_data.append({
                "id": segment.id,
                "start": segment.start,
                "end": segment.end,
                "text": segment.text.strip()
            })
            
        # Guardado estructurado en JSON
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(transcription_data, f, ensure_ascii=False, indent=4)
            
        elapsed_time = time.time() - start_time
        print(f"✔ Transcripción guardada en {json_path} (Tiempo: {elapsed_time:.2f} segundos)\n")

Modelo cargado correctamente. Iniciando procesamiento...

Transcribiendo: España - Arabia Saudí ｜ Grupo H ｜ Fase de grupos.mp3...
Detectado idioma 'es' con probabilidad 1.00
✔ Transcripción guardada en ..\data\transcripts\España - Arabia Saudí ｜ Grupo H ｜ Fase de grupos.json (Tiempo: 354.19 segundos)

Transcribiendo: España - Argentina ｜ Mundial FIFA 2026.mp3...
Detectado idioma 'es' con probabilidad 1.00
✔ Transcripción guardada en ..\data\transcripts\España - Argentina ｜ Mundial FIFA 2026.json (Tiempo: 695.24 segundos)

Transcribiendo: España - Austria ｜ Dieciseisavos de final.mp3...
Detectado idioma 'es' con probabilidad 1.00
✔ Transcripción guardada en ..\data\transcripts\España - Austria ｜ Dieciseisavos de final.json (Tiempo: 397.94 segundos)

Transcribiendo: España - Bélgica ｜ Cuartos de final.mp3...
Detectado idioma 'es' con probabilidad 1.00
✔ Transcripción guardada en ..\data\transcripts\España - Bélgica ｜ Cuartos de final.json (Tiempo: 375.07 segundos)

Transcribiendo: España